# ctxc — live demo

Compaction methods on the held-out set + unit tests. Run locally from repo root, or open in Colab.

In [ ]:
import sys, subprocess
from pathlib import Path

root = Path.cwd()
if not (root / "src" / "ctxc").exists():
    root = root.parent if (root.parent / "src" / "ctxc").exists() else root
if not (root / "src" / "ctxc").exists():
    # Colab / fresh clone — set REPO to your GitHub URL after first push
    REPO = "https://github.com/Viraj-rana/ctx-compact.git"
    subprocess.check_call(["git", "clone", "--depth", "1", REPO, "ctx-compact"])
    root = Path("ctx-compact").resolve()

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", f"{root}[dev]"])
sys.path.insert(0, str(root / "src"))
print("root:", root)

In [ ]:
import subprocess, sys
from pathlib import Path

root = Path.cwd()
if not (root / "pyproject.toml").exists() and (Path("ctx-compact") / "pyproject.toml").exists():
    root = Path("ctx-compact").resolve()
r = subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=root, capture_output=True, text=True)
print(r.stdout or r.stderr)
print("exit", r.returncode)

In [ ]:
from ctxc.eval import load_jsonl, repo_data_path, sweep_budgets
from ctxc.eval.run import format_table

ex = load_jsonl(repo_data_path("bench_test.jsonl"))
rows = sweep_budgets(ex, [40, 55, 80, 120])
print(format_table(rows))

In [ ]:
from IPython.display import Image, display
from pathlib import Path

res = Path(repo_data_path("bench_test.jsonl")).parent.parent / "results"
for name in ("acc_vs_budget.png", "dy_example.png"):
    p = res / name
    print(p.name)
    display(Image(filename=str(p)))

In [ ]:
from ctxc import build_dy, golden_convo, constraint_kept, last_n, mly_select, sim_evict

trns, pin, qry = golden_convo()
bud = 55
dy = build_dy(trns)
for name, out in {
    "mly": mly_select(trns, bud, qry, pin=pin, dy=dy),
    "last_n": last_n(trns, bud, pin=pin),
    "sim_evict": sim_evict(trns, bud, qry, pin=pin, thr=0.5),
}.items():
    print(name, "kept_ok=", constraint_kept(out), "tok=", out.used_tok, "ids=", [t.id for t in out.kept])